# 00 - Setup and verify

Run this once per machine, before any stage notebook.

## VS Code

1. Install the **Remote - SSH** and **Jupyter** extensions.
2. `Ctrl+Shift+P` -> *Remote-SSH: Add New SSH Host* -> paste the `ssh -p <port> root@<host>`
   line from the Vast.ai instance card -> Connect.
3. *File -> Open Folder* -> `/workspace/BaCP`.
4. Open a notebook, then **Select Kernel -> Python Environments -> `/venv/main/bin/python`**.

That last step matters: the system `python3` has no torch at all. Selecting it is
what produces `nohup: failed to run command 'python'` and `ModuleNotFoundError: torch`.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))          # so ladder_nb is importable
import ladder_nb as nb
info = nb.setup()


## Results directory

Keep this on `/workspace` - it survives a pod **stop**, though not a **terminate**. Checkpoints are separate: `training_utils` writes them under `project/scripts/research/`, relative to the script cwd, so moving `BACP_RESULTS_DIR` does **not** move them.


In [ ]:
from pathlib import Path
r = info['results']
print('results dir :', r, '(exists)' if r.exists() else '(will be created)')
print('on /workspace:', str(r).startswith('/workspace'))
for sub in ('runs', 'logs', 'gates', 'tables', 'reference'):
    d = r / sub
    n = len(list(d.glob('*'))) if d.is_dir() else 0
    print(f'  {sub:10s} {n}')


## Verify the code before spending GPU on it


In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, '-m', 'pytest', '-q',
                    '-W', 'ignore::DeprecationWarning', '-W', 'ignore::FutureWarning'],
                   cwd=str(info['repo']), capture_output=True, text=True)
print(p.stdout[-3000:])
print('EXIT', p.returncode)


## Pre-fetch CIFAR-10

Do this once, serially. Letting eight concurrent cells each discover a missing dataset means eight simultaneous downloads.


In [ ]:
from torchvision.datasets import CIFAR10
for train in (True, False):
    CIFAR10(str(info['repo'] / 'project' / 'scripts' / 'cache'),
            train=train, download=True)
print('ready')


## The plan

What the tier schedules, and what it costs at the measured rate.


In [ ]:
import manifest as M
for tier in (0, 1):
    cells = M.cells(tier)
    rungs = sorted({c['rung'] for c in cells})
    print(f'tier {tier}: {len(cells)} cells over {len(rungs)} rungs -> {rungs}')
